In [28]:
import time
import shutil
import os
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("HealthcareStreaming") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Medical Condition", StringType(), True),  # Fixed column name
    StructField("Hospital", StringType(), True),
    StructField("Billing Amount", DoubleType(), True)  # Fixed column name
])

streaming_dir = "/content/healthcare_stream"
checkpoint_dir = "/content/healthcare_checkpoint"

shutil.rmtree(streaming_dir, ignore_errors=True)
shutil.rmtree(checkpoint_dir, ignore_errors=True)
os.makedirs(streaming_dir, exist_ok=True)

streaming_df = spark.readStream \
    .schema(schema) \
    .option("header", True) \
    .option("maxFilesPerTrigger", 1) \
    .csv(streaming_dir)

query = streaming_df.select("Name", "Age", "Gender", "Medical Condition", "Hospital", "Billing Amount").writeStream \
    .outputMode("append") \
    .format("console") \
    .option("checkpointLocation", checkpoint_dir) \
    .start()

start_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("=" * 80)
print(f"STREAMING STARTED AT: {start_time}")
print("=" * 80)

file_path = "cleaned_healthcare_dataset.csv"
df_sample = pd.read_csv(file_path, nrows=5)
print("CSV Column Names:", df_sample.columns)

df = pd.read_csv(file_path, usecols=["Name", "Age", "Gender", "Medical Condition", "Hospital", "Billing Amount"])

max_files = 100

for index, row in df.iterrows():
    if index >= max_files:
        break

    row_df = pd.DataFrame([row])
    filename = f"{streaming_dir}/data_{index}.csv"

    row_df.to_csv(filename, index=False, header=True, mode="w", sep=",", encoding="utf-8")

    print("=" * 80)
    print(f"NEW FILE ADDED: {filename}")
    print("=" * 80)

    print("FILE HEADER:")
    print(" | ".join(row_df.columns))
    print("=" * 80)

    print("FILE CONTENT:")
    print(row_df.to_string(index=False))
    print("=" * 80)

    print(f"Processed Record {index + 1}/{max_files}:")
    print(f"| Name: {row['Name']} | Age: {row['Age']} | Gender: {row['Gender']} |")
    print(f"| Medical Condition: {row['Medical Condition']} | Hospital: {row['Hospital']} | Billing Amount: ${row['Billing Amount']:.2f} |")
    print("=" * 80)

    time.sleep(10)

print("=" * 80)
print("STREAMING STOPPED AFTER PROCESSING 100 RECORDS.")
print("=" * 80)

query.stop()


STREAMING STARTED AT: 2025-03-26 15:05:08
CSV Column Names: Index(['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition',
       'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider',
       'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date',
       'Medication', 'Test Results'],
      dtype='object')
NEW FILE ADDED: /content/healthcare_stream/data_0.csv
FILE HEADER:
Name | Age | Gender | Medical Condition | Hospital | Billing Amount
FILE CONTENT:
     Name  Age Gender Medical Condition Hospital  Billing Amount
Carl Best   60   male      Hypertension Ltd Wang     26062.43432
Processed Record 1/100:
| Name: Carl Best | Age: 60 | Gender: male |
| Medical Condition: Hypertension | Hospital: Ltd Wang | Billing Amount: $26062.43 |
NEW FILE ADDED: /content/healthcare_stream/data_1.csv
FILE HEADER:
Name | Age | Gender | Medical Condition | Hospital | Billing Amount
FILE CONTENT:
            Name  Age Gender Medical Condition       Hospital  Billing Amount
Josep

KeyboardInterrupt: 